# FIK FAIR v12: stability-focused residual correction

Attach official competition data and the actual `submission_v11.csv`; optionally attach `v11_oof.csv` for paired validation. CPU is sufficient. The user reports v11 public RMSE 1.9549491170; it is not used in fitting.

The new component uses boundary-corrected local-linear trends and feature residual × rank interactions. Inner CV chooses between the original and new component. Final weights are fixed: 50% v11, 50% selected component. Permission to use file order is reported by the team, not independently verified. Preserve input order and dependency versions for reproduction.

In [1]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.special import betainc
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold, RepeatedKFold

ON_KAGGLE = Path('/kaggle/input').exists()
# Set explicit paths here if multiple candidate inputs are attached.
DATA_DIR = None
V11_PATH = None
V11_OOF_PATH = None
if ON_KAGGLE:
    input_root = Path('/kaggle/input')
    if DATA_DIR is None:
        paths = []
        for path in input_root.rglob('train.csv'):
            columns = pd.read_csv(path, nrows=0).columns
            if {'record_id', 'target', 'feature_24'} <= set(columns) and (path.parent / 'test.csv').exists():
                paths.append(path.parent)
        if len(paths) != 1:
            raise ValueError(f'Set DATA_DIR explicitly; found {paths}')
        DATA_DIR = paths[0]
    if V11_PATH is None:
        paths = list(input_root.rglob('submission_v11.csv'))
        if len(paths) != 1:
            raise ValueError(f'Attach only the v11 submission, or set V11_PATH explicitly: {paths}')
        V11_PATH = paths[0]
    if V11_OOF_PATH is None:
        paths = list(input_root.rglob('v11_oof.csv'))
        if len(paths) == 1:
            V11_OOF_PATH = paths[0]
    OUT = Path('/kaggle/working')
else:
    root = Path.cwd()
    if not (root / 'data/train.csv').exists():
        root = root.parent
    DATA_DIR = Path(DATA_DIR) if DATA_DIR is not None else root / 'data'
    V11_PATH = Path(V11_PATH) if V11_PATH is not None else root / 'outputs/fikfair_v11/submission_v11.csv'
    V11_OOF_PATH = Path(V11_OOF_PATH) if V11_OOF_PATH is not None else root / 'outputs/stability_search/v11_oof.csv'
    OUT = root / 'outputs/fikfair_v12'
OUT.mkdir(parents=True, exist_ok=True)
DATA_DIR, V11_PATH = Path(DATA_DIR), Path(V11_PATH)
print('Data:', DATA_DIR, '\nv11:', V11_PATH, '\nOutput:', OUT)

from sklearn.linear_model import Ridge
import scipy
import sklearn
import sys
import hashlib
import platform
VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__}
print('Environment:', VERSIONS)


Data: /home/rian/Projects/fikfair/data 
v11: /home/rian/Projects/fikfair/outputs/fikfair_v11/submission_v11.csv 
Output: /home/rian/Projects/fikfair/outputs/fikfair_v12
Environment: {'python': '3.12.14', 'numpy': '2.5.3', 'pandas': '3.0.6', 'scipy': '1.18.1', 'scikit-learn': '1.9.1'}


In [2]:
tr = pd.read_csv(DATA_DIR / 'train.csv')
te = pd.read_csv(DATA_DIR / 'test.csv')
template = pd.read_csv(DATA_DIR / 'submission_template.csv')
FEATURES = [c for c in tr.columns if c.startswith('feature_')]
X, y, Xte = tr[FEATURES].to_numpy(), tr.target.to_numpy(), te[FEATURES].to_numpy()
assert len(FEATURES) == 24
assert np.isfinite(X).all() and np.isfinite(Xte).all() and np.isfinite(y).all()
assert tr.record_id.is_unique and te.record_id.is_unique
assert not set(tr.record_id) & set(te.record_id)
v11_file = pd.read_csv(V11_PATH)
assert list(v11_file.columns) == ['record_id', 'prediction']
assert len(v11_file) == len(te) and v11_file.record_id.is_unique
assert set(v11_file.record_id) == set(te.record_id)
v11_test = te[['record_id']].merge(v11_file, on='record_id', validate='one_to_one').prediction.to_numpy()
assert np.isfinite(v11_test).all()
# Baseline is the exact v11 artifact from the preceding experiment.
# Checking known predictions prevents accidentally attaching another version.
expected_head = np.array([93.85012407910526, 85.30611538214416, 78.67159041429093, 73.96624977832471, 70.79465537287832])
assert np.allclose(v11_test[:5], expected_head, atol=2e-5, rtol=0), 'This is not the v11 submission audited here'
baseline_oof = None
if V11_OOF_PATH is not None and Path(V11_OOF_PATH).exists():
    saved = pd.read_csv(V11_OOF_PATH)
    assert saved.record_id.is_unique and set(saved.record_id) == set(tr.record_id)
    saved = tr[['record_id']].merge(saved, on='record_id', validate='one_to_one')
    assert np.allclose(saved.target, y)
    baseline_oof = saved[['v11_rep1', 'v11_rep2', 'v11_rep3']].to_numpy().T
print('Train/test:', X.shape, Xte.shape, '| archived baseline OOF:', baseline_oof is not None)


Train/test: (382, 24) (96, 24) | archived baseline OOF: True


## Two residual candidates

Both use the same training-only PCA-neighbor rank correction and target curve. The old component removes Gaussian-smoothed feature/target trends. The new component uses local-linear trends, reducing boundary bias, and includes residual feature × centered-rank interactions so coefficients may vary along the ordered file. Ridge alpha stays fixed at 10.

Inner CV chooses one of these two. Neither validation labels nor public scores fit transformations or blend weights. Candidate discovery used the same dataset, so research-selection bias remains.

In [3]:
def rank(n):
    return (np.arange(n)+.5)/n

def curve(t, u, smooth=0):
    q=rank(len(t))
    ys=IsotonicRegression(increasing=False).fit_transform(q,t)
    if smooth:
        ys=gaussian_filter1d(ys, smooth*len(t),mode='nearest')
    return np.interp(u,q,ys)

def neighbor_signals(A,B,k=10,dim=12):
    mu,sd=A.mean(0),A.std(0)
    pc=PCA(n_components=dim,svd_solver='full').fit((A-mu)/sd)
    ZA=pc.transform((A-mu)/sd)/np.sqrt(pc.explained_variance_)
    ZB=pc.transform((B-mu)/sd)/np.sqrt(pc.explained_variance_)
    DA=cdist(ZA,ZA);np.fill_diagonal(DA,np.inf)
    DB=cdist(ZB,ZA)
    qa=rank(len(A))
    def local(D):
        ix=np.argpartition(D,k-1,axis=1)[:,:k]
        weights=1/np.maximum(np.take_along_axis(D,ix,axis=1),.05)**2
        return (weights*qa[ix]).sum(1)/weights.sum(1)
    return local(DA),local(DB)

def warp(A,B,k=10,dim=12,kind='gaussian',strength=1.):
    ref,query=neighbor_signals(A,B,k,dim)
    qa,qb=rank(len(A)),rank(len(B))
    trend=gaussian_filter1d(ref,.05*len(A),mode='nearest')
    error=query-np.interp(qb,qa,trend)
    if kind=='bridge':
        # Gaussian approximation to train/query empirical rank displacement.
        # min(u,v)-u*v is the Brownian bridge covariance kernel.
        K=(np.minimum.outer(qb,qb)-np.outer(qb,qb))*(1/len(A)+1/len(B))*strength
        derivative=np.clip(np.interp(qb,qa,np.gradient(trend,qa)),.1,3.)
        sigma2=max(float(np.mean((ref-trend)**2)),1e-4)
        cov=K*derivative[:,None]*derivative[None,:]+np.eye(len(B))*sigma2
        correction=(K*derivative[None,:])@np.linalg.solve(cov,error)
    else:
        correction=strength*4*qb*(1-qb)*gaussian_filter1d(error,max(1,.1*len(B)),mode='nearest')
    return IsotonicRegression(increasing=True).fit_transform(qb,np.clip(qb+correction,0,1))

def local_linear(z,h=.05):
    q=rank(len(z));dx=q[None,:]-q[:,None]
    w=np.exp(-.5*(dx/h)**2)
    s0=w.sum(1);s1=(w*dx).sum(1);s2=(w*dx**2).sum(1)
    S=w*(s2[:,None]-s1[:,None]*dx)/(s0*s2-s1*s1)[:,None]
    return S@z

def all_candidates(A,t,B):
    qa=rank(len(A));u=warp(A,B)
    base=curve(t,u,.005)
    mu,sd=A.mean(0),A.std(0);Z=(A-mu)/sd;Q=(B-mu)/sd
    out={}
    for smoother in ['gaussian','local_linear']:
        if smoother=='gaussian':
            tx=gaussian_filter1d(Z,.05*len(A),axis=0,mode='nearest')
            ty=gaussian_filter1d(t,.05*len(A),mode='nearest')
        else:tx,ty=local_linear(Z),local_linear(t)
        RX=Z-tx;RQ=Q-np.column_stack([np.interp(u,qa,tx[:,j]) for j in range(Z.shape[1])])
        for interaction in [False,True]:
            RA=np.c_[RX,RX*(2*qa[:,None]-1)] if interaction else RX
            RB=np.c_[RQ,RQ*(2*u[:,None]-1)] if interaction else RQ
            for alpha in [10,100]:
                p=base+Ridge(alpha=alpha).fit(RA,t-ty).predict(RB)
                key=f'{smoother}_interaction{int(interaction)}_ridge{alpha}'
                out[key]=p
    baseline=out['gaussian_interaction0_ridge10']
    # A fixed conservative hedge for each extension.
    for key,p in list(out.items()):
        if key!='gaussian_interaction0_ridge10':out[key+'_half']=.5*(p+baseline)
    return out

In [4]:
KEYS = ['gaussian_interaction0_ridge10', 'local_linear_interaction1_ridge10']

def candidates(A,t,B):
    all_p=all_candidates(A,t,B)
    return {k:all_p[k] for k in KEYS}

def select_rule(A,t,seed):
    losses={k:0. for k in KEYS}
    for a,b in KFold(5,shuffle=True,random_state=seed).split(A):
        a,b=np.sort(a),np.sort(b)
        assert not set(a)&set(b)
        for key,p in candidates(A[a],t[a],A[b]).items():
            losses[key]+=np.sum((p-t[b])**2)
    return min(losses,key=losses.get),{k:float(np.sqrt(v/len(t))) for k,v in losses.items()}

## Nested validation

The 15 outer folds match the archived v11 predictions. For each fold, candidate selection uses only the outer training rows. All transforms, smoothing trends and residual models are fitted there. The final blend weight remains 50:50.

Optional v11 OOF is the prior nested local audit baseline, not a claim of bitwise identical Kaggle predictions. The baseline and candidate catalogue were developed on this same dataset; results remain development estimates.

In [5]:
start = time.perf_counter()
folds = list(RepeatedKFold(n_splits=5, n_repeats=3, random_state=42).split(X))
rank_oof, quantile_oof = np.zeros((3, len(y))), np.zeros((3, len(y)))
fold_rows = []
for i, (a, b) in enumerate(folds):
    a, b = np.sort(a), np.sort(b)
    assert not set(a) & set(b)
    selected, inner_scores = select_rule(X[a], y[a], 560100 + i)
    predictions = candidates(X[a], y[a], X[b])
    rank_oof[i // 5, b] = predictions[selected]
    quantile_oof[i // 5, b] = curve(y[a], rank(len(b)))
    fold_rows.append({'fold': i, 'selected': selected, 'inner_rmse': inner_scores[selected]})
    print(f'Fold {i+1}/15: {selected}')

predictions = {'quantile': quantile_oof, 'nested_rank': rank_oof}
if baseline_oof is not None:
    predictions['v11'] = baseline_oof
    predictions['v12'] = .5 * (baseline_oof + rank_oof)
rows = []
for name, p in predictions.items():
    fs = [np.sqrt(np.mean((p[i//5, b] - y[b])**2)) for i, (_, b) in enumerate(folds)]
    reps = [np.sqrt(np.mean((pp-y)**2)) for pp in p]
    rows.append({'method': name, 'mean_fold_rmse': np.mean(fs), 'mean_repeat_rmse': np.mean(reps),
                 'repeat1': reps[0], 'repeat2': reps[1], 'repeat3': reps[2]})
scores = pd.DataFrame(rows)
print(scores.round(4).to_string(index=False))
print(f'CPU diagnostic elapsed: {time.perf_counter()-start:.1f} seconds')
scores.to_csv(OUT / 'validation.csv', index=False)
pd.DataFrame(fold_rows).to_csv(OUT / 'fold_selection.csv', index=False)
np.savez(OUT / 'validation_predictions.npz', **predictions)


Fold 1/15: local_linear_interaction1_ridge10
Fold 2/15: local_linear_interaction1_ridge10
Fold 3/15: local_linear_interaction1_ridge10
Fold 4/15: local_linear_interaction1_ridge10
Fold 5/15: local_linear_interaction1_ridge10
Fold 6/15: local_linear_interaction1_ridge10
Fold 7/15: local_linear_interaction1_ridge10
Fold 8/15: local_linear_interaction1_ridge10
Fold 9/15: local_linear_interaction1_ridge10
Fold 10/15: local_linear_interaction1_ridge10
Fold 11/15: local_linear_interaction1_ridge10
Fold 12/15: local_linear_interaction1_ridge10
Fold 13/15: local_linear_interaction1_ridge10
Fold 14/15: local_linear_interaction1_ridge10
Fold 15/15: local_linear_interaction1_ridge10
     method  mean_fold_rmse  mean_repeat_rmse  repeat1  repeat2  repeat3
   quantile          3.6479            3.7037   3.6786   3.7756   3.6569
nested_rank          3.1923            3.2307   3.1058   3.4173   3.1689
        v11          3.3031            3.3405   3.1877   3.5095   3.3243
        v12          3.1789

## Export and provenance

Choose one candidate using five-fold CV on all training rows, then average its test prediction with the exact v11 input file. A manifest records source-file hashes, dependency versions, selected model, seed, and output hash. The export is rerun to verify byte-identical CSV output in this environment. No submission is uploaded automatically.

In [6]:
selected, inner_scores = select_rule(X, y, 560199)
new_test = candidates(X, y, Xte)[selected]
final_test = .5 * (v11_test + new_test)
assert np.isfinite(final_test).all()
assert set(template.record_id) == set(te.record_id) and template.record_id.is_unique

def submission_frame(p):
    result = template[['record_id']].merge(
        pd.DataFrame({'record_id': te.record_id, 'prediction': p}),
        on='record_id', how='left', validate='one_to_one')
    assert len(result) == len(te) and np.isfinite(result.prediction).all()
    return result

result = submission_frame(final_test)
csv_text = result.to_csv(index=False)
output_path = OUT / 'submission_v12.csv'
output_path.write_text(csv_text)
repeat_selected, _ = select_rule(X, y, 560199)
repeat_new = candidates(X, y, Xte)[repeat_selected]
repeat_text = submission_frame(.5 * (v11_test + repeat_new)).to_csv(index=False)
assert selected == repeat_selected and csv_text == repeat_text

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

manifest = {
    'selected': selected, 'full_training_cv': inner_scores,
    'v11_weight': .5, 'new_model_weight': .5, 'selection_seed': 560199,
    'versions': VERSIONS, 'platform': platform.platform(),
    'input_sha256': {str(p): sha256_file(p) for p in [DATA_DIR/'train.csv', DATA_DIR/'test.csv',
                     DATA_DIR/'submission_template.csv', V11_PATH]},
    'submission_sha256': sha256_file(output_path),
    'same_environment_repeat_identical': True,
    'v11_public_reported_by_user': 1.9549491170, 'v12_public': None,
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
(OUT / 'requirements-v12.txt').write_text('\n'.join(f'{k}=={v}' for k,v in VERSIONS.items() if k != 'python')+'\n')
changes = pd.DataFrame({'record_id': te.record_id, 'v11': v11_test,
                       'new_model': new_test, 'v12': final_test, 'change': final_test-v11_test})
changes.to_csv(OUT / 'prediction_changes.csv', index=False)
print('Selected:', selected)
print('Repeated export is byte-identical:', True)
print('RMS prediction change:', np.sqrt(np.mean((final_test-v11_test)**2)))
print('Submission SHA256:', manifest['submission_sha256'])
print(result.head().to_string(index=False))


Selected: local_linear_interaction1_ridge10
Repeated export is byte-identical: True
RMS prediction change: 0.5544188418943297
Submission SHA256: f4f24e0cf5aa480b77d724f82b2befe69267492dfd06dcb352cff8eda11cb462
record_id  prediction
    D0263   94.691586
    D0280   86.177338
    D0206   79.156973
    D0133   74.314974
    D0352   70.223353


## Evidence and limits

Local nested development RMSE: v11 3.3031; fixed 50:50 refinement 3.1789; improvement in 14/15 folds and all three repeats. Record-cluster bootstrap delta interval [-0.1871, -0.0538], conditional on stored predictions.

On 200 additional 96-row query partitions, the new standalone component improved mean RMSE 3.4692 → 3.3586 and simulated private RMSE p95 4.6794 → 4.5641. Those are component comparisons, not the full v11/v12 ensemble. Each query was divided into disjoint 29/67 subsets. These are repeated partitions of the same 382 records, not independent datasets or a forecast of Kaggle private RMSE. Public/private equality cannot be guaranteed.

Same-environment repeat export is checked below; cross-hardware reproducibility and generalization are separate questions. End-to-end reproduction also requires the v9 → v10 → v11 baseline pipeline and its artifacts.

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rank(len(te)), v11_test, label='v11')
axes[0].plot(rank(len(te)), new_test, label='rank + residual', alpha=.8)
axes[0].plot(rank(len(te)), final_test, label='v12')
axes[0].set(xlabel='Position within test file', ylabel='Prediction', title='Final candidate predictions')
axes[0].legend()
axes[1].plot(rank(len(te)), final_test-v11_test)
axes[1].axhline(0, color='black', lw=1)
axes[1].set(xlabel='Position within test file', ylabel='v12 minus v11', title='Changes do not use test labels')
fig.tight_layout()
fig.savefig(OUT / 'v12_predictions.png', dpi=150)
print('Saved plot:', OUT / 'v12_predictions.png')


Saved plot: /home/rian/Projects/fikfair/outputs/fikfair_v12/v12_predictions.png
